# 03. Remuestreo conjunto uniforme

Esta notebook construye un primer modelo de referencia de una sesión mediante muestreo con reemplazo de observaciones históricas conjuntas. El objetivo es hacer explícitos sus supuestos, no validarlo como modelo del futuro ni presentar una implementación completa de Fully Flexible Resampling (FFR).

El alcance se limita a la preparación autónoma de las entradas y al remuestreo conjunto uniforme con horizonte $H=1$.

In [1]:
from __future__ import annotations

import hashlib
import json
import site
import sys
from datetime import date
from pathlib import Path

import numpy as np

def find_project_root() -> Path:
    def is_project_root(candidate: Path) -> bool:
        return ((candidate / 'config' / 'universe.csv').is_file()
                and (candidate / 'src' / 'scenario_portfolio_lab').is_dir())

    for origin in (Path.cwd(), *Path.cwd().parents):
        for candidate in (origin, origin / 'scenario-portfolio-lab', origin / 'data-science-portfolio' / 'scenario-portfolio-lab'):
            if is_project_root(candidate):
                return candidate
    raise RuntimeError(
        'Repository files config/universe.csv and src/scenario_portfolio_lab were not found. '
        'Open this notebook from the complete scenario-portfolio-lab repository (or its data-science-portfolio parent). '
        'Do not run it from an exported standalone .ipynb file.'
    )


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# MyST/Jupyter kernels may differ from the project virtual environment.
# Reuse already installed local dependencies when this Windows project .venv exists; never install packages here.
LOCAL_SITE_PACKAGES = ROOT / '.venv' / 'Lib' / 'site-packages'
if LOCAL_SITE_PACKAGES.is_dir():
    site.addsitedir(str(LOCAL_SITE_PACKAGES))

try:
    import exchange_calendars as xcals
    import polars as pl
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'Missing project dependencies. Select the project .venv Python 3.13.2 kernel or install requirements.txt '
        'into the active kernel environment before running this notebook.'
    ) from exc

RUN_ROOT = ROOT / 'data' / 'published' / 'cut_2026-10-09_run_2026-10-11_full_window_final'
GOLD_ROOT = RUN_ROOT / 'gold'
PRICE_MATRIX_PATH = GOLD_ROOT / 'adjusted_close_matrix.parquet'
RETURN_MATRIX_PATH = GOLD_ROOT / 'eligible_simple_total_return_matrix.parquet'
RUN_MANIFEST_PATH = RUN_ROOT / 'run_manifest.json'
ELIGIBLE_CATALOG_PATH = RUN_ROOT / 'reports' / 'eligible_universe.csv'

for required in (PRICE_MATRIX_PATH, RETURN_MATRIX_PATH, RUN_MANIFEST_PATH, ELIGIBLE_CATALOG_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'Required final-cut artifact is missing: {required.relative_to(ROOT)}')


## Entradas autónomas y contrato

La notebook reutiliza el mismo corte final, catálogo elegible y convenciones de la notebook 02: 125 instrumentos, 2,205 precios ajustados y 2,204 retornos diarios utilizables. Sólo se retira la primera fila nula prevista de retornos; no se imputan ni se filtran observaciones.

Se validan fechas, orden de símbolos, dominio de `log1p` e integridad de los raw referenciados en el manifiesto. El manifiesto no contiene hashes de referencia para los dos Parquet matriciales; por eso, estas comprobaciones de contenido no equivalen a verificar históricamente esas matrices contra hashes de referencia.

In [2]:
run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding='utf-8'))
approved_period = run_manifest['approved_period']
assert approved_period == {'start': '2018-01-02', 'end': '2026-10-09'}

eligible_catalog = pl.read_csv(ELIGIBLE_CATALOG_PATH).filter(pl.col('universe_status') == 'ELIGIBLE')
eligible_symbols = eligible_catalog['symbol'].to_list()
assert len(eligible_symbols) == 125 and len(set(eligible_symbols)) == 125

raw_hash_checks = []
for symbol, reference in run_manifest['raw'].items():
    raw_path = ROOT / Path(reference['path'].replace('\\', '/'))
    if not raw_path.is_file():
        raise FileNotFoundError(f'Manifest-referenced raw is missing for {symbol}.')
    raw_hash_checks.append(hashlib.sha256(raw_path.read_bytes()).hexdigest() == reference['sha256'])
assert len(raw_hash_checks) == 138 and all(raw_hash_checks)

all_gold_prices = pl.read_parquet(PRICE_MATRIX_PATH)
return_matrix = pl.read_parquet(RETURN_MATRIX_PATH)
assert all_gold_prices.columns[0] == return_matrix.columns[0] == 'session_date'
return_symbols = return_matrix.columns[1:]
assert return_symbols == eligible_symbols
eligible_prices = all_gold_prices.select(['session_date', *return_symbols])
assert eligible_prices.shape == return_matrix.shape == (2205, 126)

start, end = date.fromisoformat(approved_period['start']), date.fromisoformat(approved_period['end'])
expected_dates = [stamp.date() for stamp in xcals.get_calendar('XNYS').sessions if start <= stamp.date() <= end]
for matrix in (eligible_prices, return_matrix):
    matrix_dates = matrix['session_date'].to_list()
    assert matrix_dates == expected_dates
    assert matrix_dates == sorted(matrix_dates) and len(set(matrix_dates)) == len(matrix_dates) == 2205

price_values = eligible_prices.select(pl.exclude('session_date'))
assert all(value == 0 for value in price_values.null_count().row(0))
assert not any(price_values.select(pl.all().is_nan().any()).row(0))
assert not any(price_values.select(pl.all().is_infinite().any()).row(0))
assert not any(price_values.select((pl.all() <= 0).any()).row(0))

first_return_row = return_matrix.slice(0, 1).select(pl.exclude('session_date'))
subsequent_return_rows = return_matrix.slice(1).select(pl.exclude('session_date'))
assert all(value == 1 for value in return_matrix.select(pl.exclude('session_date').null_count()).row(0))
assert all(value == 1 for value in first_return_row.null_count().row(0))
assert all(value == 0 for value in subsequent_return_rows.null_count().row(0))
assert not any(subsequent_return_rows.select(pl.all().is_nan().any()).row(0))
assert not any(subsequent_return_rows.select(pl.all().is_infinite().any()).row(0))

usable_returns = return_matrix.slice(1).select(['session_date', *return_symbols])
assert usable_returns.shape == (2204, 126)
assert usable_returns['session_date'].to_list() == expected_dates[1:]
simple_return_values = usable_returns.select(pl.exclude('session_date')).to_numpy()
domain_failures = np.argwhere(simple_return_values <= -1.0)
if domain_failures.size:
    examples = [{'session_date': str(usable_returns['session_date'][int(i)]), 'symbol': return_symbols[int(j)], 'simple_return': float(simple_return_values[i, j])} for i, j in domain_failures[:10]]
    raise ValueError(f'Log-return domain failure (r <= -1); first locations: {examples}')
log_return_values = np.log1p(simple_return_values)
if not np.isfinite(log_return_values).all():
    raise ValueError('Log-return transformation produced a non-finite value.')

historical_dates = np.asarray(usable_returns['session_date'].to_list())
T, N = log_return_values.shape
assert (T, N) == (2204, 125)
historical_probabilities = np.full(T, 1.0 / T, dtype=float)
assert np.isfinite(historical_probabilities).all() and (historical_probabilities > 0.0).all()
assert np.isclose(float(historical_probabilities.sum()), 1.0, rtol=0.0, atol=1e-12)

input_validation_summary = pl.DataFrame({
    'check': ['Approved period', 'Eligible symbols', 'Price matrix', 'Usable simple-return matrix', 'Log-return matrix', 'Manifest-referenced raw hashes', 'Matrix reference hashes'],
    'result': ['2018-01-02 to 2026-10-09', '125 exact symbols, explicit order', '2,205 rows × (date + 125); positive, finite, no nulls', '2,204 rows × (date + 125); first null row removed only', '2,204 rows × 125; finite and log1p domain valid', '138/138 verified against manifest references', 'Not available in run_manifest; no new hash is treated as prior evidence'],
})
input_validation_summary


check,result
str,str
"""Approved period""","""2018-01-02 to 2026-10-09"""
"""Eligible symbols""","""125 exact symbols, explicit or…"
"""Price matrix""","""2,205 rows × (date + 125); pos…"
"""Usable simple-return matrix""","""2,204 rows × (date + 125); fir…"
"""Log-return matrix""","""2,204 rows × 125; finite and l…"
"""Manifest-referenced raw hashes""","""138/138 verified against manif…"
"""Matrix reference hashes""","""Not available in run_manifest;…"


## 1. Remuestreo conjunto uniforme a una sesión

Con probabilidades históricas uniformes, se define:

$$I_s \sim \mathrm{Categorical}(p_1,\ldots,p_T), \qquad \ell_s^*=\ell_{I_s}, \qquad r_s^*=\mathrm{expm1}(\ell_s^*).$$

Cada índice selecciona una sesión completa y, con ella, los 125 retornos simultáneos. La masa histórica $p_t$ pertenece a la distribución de origen; la frecuencia observada de un índice en una simulación finita puede diferir de ella. Cada realización generada recibe además un peso Monte Carlo de $1/S$.

La fecha seleccionada identifica la observación histórica de origen; no es una fecha futura.

In [3]:
HORIZON_SESSIONS = 1
SIMULATION_COUNT = 10_000
RNG_SEED = 20_261_009
RNG_NAME = 'NumPy Generator(PCG64)'
RECOVERY_ABSOLUTE_TOLERANCE, RECOVERY_RELATIVE_TOLERANCE = 1e-12, 1e-10

rng = np.random.Generator(np.random.PCG64(RNG_SEED))
selected_indices = rng.choice(T, size=SIMULATION_COUNT, replace=True, p=historical_probabilities)
selected_source_dates = historical_dates[selected_indices]
selected_log_returns = log_return_values[selected_indices, :]
selected_simple_returns = np.expm1(selected_log_returns)
selected_original_simple_returns = simple_return_values[selected_indices, :]
monte_carlo_weights = np.full(SIMULATION_COUNT, 1.0 / SIMULATION_COUNT, dtype=float)
selection_frequencies = np.bincount(selected_indices, minlength=T)

EXPOSURE_EXAMPLES = ('SPY', 'TLT', 'GLD')
missing_examples = [symbol for symbol in EXPOSURE_EXAMPLES if symbol not in return_symbols]
if missing_examples:
    raise RuntimeError(f'Los ejemplos no están en el universo elegible; no se hizo sustitución: {missing_examples}')
example_indices = [return_symbols.index(symbol) for symbol in EXPOSURE_EXAMPLES]

replayed_rng = np.random.Generator(np.random.PCG64(RNG_SEED))
replayed_indices = replayed_rng.choice(T, size=SIMULATION_COUNT, replace=True, p=historical_probabilities)

resampling_checks = {
    'Exactamente S índices enteros en el rango válido': selected_indices.shape == (SIMULATION_COUNT,) and np.issubdtype(selected_indices.dtype, np.integer) and bool(((selected_indices >= 0) & (selected_indices < T)).all()),
    'Dimensiones S × 125': selected_log_returns.shape == selected_simple_returns.shape == (SIMULATION_COUNT, N),
    'Orden explícito de símbolos conservado': tuple(return_symbols) == tuple(eligible_symbols) and len(return_symbols) == N,
    'Cada vector logarítmico coincide con su fila histórica de origen': bool(np.array_equal(selected_log_returns, log_return_values[selected_indices, :])),
    'Fechas de origen alineadas con los índices': bool(np.array_equal(selected_source_dates, historical_dates[selected_indices])),
    'Retornos simples recuperados coinciden con los originales': bool(np.allclose(selected_simple_returns, selected_original_simple_returns, rtol=RECOVERY_RELATIVE_TOLERANCE, atol=RECOVERY_ABSOLUTE_TOLERANCE)),
    'Retornos seleccionados finitos y dentro del dominio': bool(np.isfinite(selected_log_returns).all() and np.isfinite(selected_simple_returns).all() and (selected_simple_returns > -1.0).all()),
    'Frecuencias de selección suman S': selection_frequencies.shape == (T,) and int(selection_frequencies.sum()) == SIMULATION_COUNT,
    'Pesos Monte Carlo positivos y con suma uno': bool(np.isfinite(monte_carlo_weights).all() and (monte_carlo_weights > 0.0).all() and np.isclose(float(monte_carlo_weights.sum()), 1.0, rtol=0.0, atol=1e-12)),
    'Misma configuración reproduce los mismos índices': bool(np.array_equal(selected_indices, replayed_indices)),
}
if not all(resampling_checks.values()):
    failed_checks = [name for name, passed in resampling_checks.items() if not passed]
    raise AssertionError(f'La validación de remuestreo conjunto falló sin alterar los datos: {failed_checks}')

simulation_preview = pl.DataFrame({
    'simulation_id': np.arange(5, dtype=int),
    'historical_index': selected_indices[:5],
    'source_date': selected_source_dates[:5],
    **{f'{symbol}_simple_return': selected_simple_returns[:5, index] for symbol, index in zip(EXPOSURE_EXAMPLES, example_indices)},
})
resampling_configuration = pl.DataFrame({
    'item': ['Historical observations (T)', 'Instruments (N)', 'Simulations (S)', 'Horizon (H)', 'Seed', 'Generator', 'Historical probability', 'Monte Carlo weight', 'Distinct historical observations selected'],
    'value': [str(T), str(N), str(SIMULATION_COUNT), '1 session', str(RNG_SEED), RNG_NAME, f'1/{T} = {historical_probabilities[0]:.18f}', f'1/{SIMULATION_COUNT} = {monte_carlo_weights[0]:.10f}', str(int((selection_frequencies > 0).sum()))],
})
resampling_check_summary = pl.DataFrame({'check': list(resampling_checks), 'result': ['PASS' if passed else 'FAIL' for passed in resampling_checks.values()]})

resampling_configuration, simulation_preview, resampling_check_summary


(shape: (9, 2)
 ┌─────────────────────────────────┬───────────────────────────────┐
 │ item                            ┆ value                         │
 │ ---                             ┆ ---                           │
 │ str                             ┆ str                           │
 ╞═════════════════════════════════╪═══════════════════════════════╡
 │ Historical observations (T)     ┆ 2204                          │
 │ Instruments (N)                 ┆ 125                           │
 │ Simulations (S)                 ┆ 10000                         │
 │ Horizon (H)                     ┆ 1 session                     │
 │ Seed                            ┆ 20261009                      │
 │ Generator                       ┆ NumPy Generator(PCG64)        │
 │ Historical probability          ┆ 1/2204 = 0.000453720508166969 │
 │ Monte Carlo weight              ┆ 1/10000 = 0.0001000000        │
 │ Distinct historical observatio… ┆ 2183                          │
 └─────────────────

## Interpretación y límites

Seleccionar vectores completos conserva la estructura conjunta de la distribución empírica de origen. Una simulación finita, sin embargo, presenta error Monte Carlo. Usarla como distribución futura supone que la historia observada, con pesos uniformes, es representativa. Este modelo no condiciona en el estado actual ni favorece observaciones recientes.

Las realizaciones se sortean independientemente: esa es una decisión del simulador, no una propiedad demostrada de los datos. Como $H=1$, todavía no se modela dependencia entre sesiones de una trayectoria. El soporte de los vectores diarios queda limitado a observaciones históricas; aumentar $S$ reduce error Monte Carlo, pero no aporta información histórica nueva ni resuelve errores del modelo.

No se implementan aún trayectorias multisesión, ponderación temporal, estados, FFR, Entropy Pooling, carteras, VaR/CVaR, optimización ni backtest. Las filas mostradas son realizaciones independientes de una sesión, no una trayectoria temporal.